# 🔑 ISOM 260: Your First API Call

**Session 3 — Three Secrets & Your First API Key** | Suffolk University | Prof. Hasan Arslan

---

Until today, you've *visited* AI — typing into someone else's chat window. Today your **own code** talks to a frontier model. This is the moment the rest of the course (agents, tools, RAG, your final project) is built on.

### Before you run anything
1. Get your free API key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey) (sign in with your **personal** Google account)
2. In Colab, click the **🔑 key icon** in the left sidebar → **Add new secret** → Name: `GOOGLE_API_KEY` → paste your key → toggle **Notebook access** ON
3. `File → Save a copy in Drive`, then run cells top to bottom

> 🔒 **The one rule of API keys:** treat it like a password. Never paste it into code you might share, never screenshot it. The Secrets panel exists exactly for this.


In [ ]:
# ── Install the Google Gen AI Python SDK ──────────────────────────
!pip install -q google-genai

In [ ]:
# ── Load your API key from Colab Secrets ──────────────────────────
# Get yours at: https://aistudio.google.com/apikey
from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    print("✅ API key loaded from Colab Secrets!")
except Exception:
    GOOGLE_API_KEY = "your-api-key-here"   # <-- last resort only; Secrets is the right way
    print("⚠️ Using hardcoded API key. Use the 🔑 Secrets panel instead.")

## 🚀 First #1 — The call

Six lines. Read them — this is the entire shape of every AI product you'll ever build: *client → model → prompt → response.*


In [ ]:
# ── Your first API call ───────────────────────────────────────────
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="In exactly one sentence, congratulate a business student on their first-ever API call.",
)

print(response.text)
print("\n🎉 That answer did not come from a chat window. YOUR code requested it.")

**What just happened, technically:** your code sent a request over the internet → a Google datacenter ran a frontier model's *inference* on it → the generated tokens came back to your notebook. Round trip: about a second.

## 🌡️ First #2 — The temperature dial (you already know this one)

In Session 1 you built this knob yourself with the name generator. Same knob, frontier model:


In [ ]:
# ── The creativity dial, on a real model ──────────────────────────
from google.genai import types

prompt = "Suggest a name for a coffee shop inside a business school. Name only."

for temp in [0.1, 1.0, 1.9]:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temp,
            thinking_config=types.ThinkingConfig(thinking_budget=0),  # raw dial only — no hidden "thinking" pass
        ),
    )
    label = {0.1: "😴 T=0.1 (safe)   ", 1.0: "🙂 T=1.0 (normal) ", 1.9: "🤪 T=1.9 (chaotic)"}[temp]
    print(label, "→", response.text.strip())

# 🎮 YOUR TURN: re-run this cell — do the T=0.1 answers repeat? Why?


Low temperature = the model takes the tallest probability bar almost every time (remember the crystal ball). High temperature = it gambles. **Business instinct check:** invoice extraction — high or low? Campaign brainstorming?

## 🧾 First #3 — Count the tokens, price the call

Secret #1 of today's lecture: everything is billed in **tokens**. Let's see the meter actually running:


In [ ]:
# ── The meter is running: tokens in, tokens out ───────────────────
business_prompt = """Summarize in 3 bullet points why a coffee subscription
service might struggle in an office building that already has free coffee."""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=business_prompt,
)
print(response.text)

usage = response.usage_metadata
tokens_in  = usage.prompt_token_count or 0
tokens_out = usage.candidates_token_count or 0
thinking   = usage.thoughts_token_count or 0   # the model "thinks" before answering — hidden, but billed as output
print("─" * 50)
print(f"tokens IN   (your prompt):        {tokens_in}")
print(f"tokens OUT  (the answer):         {tokens_out}")
print(f"tokens THINKING (hidden work):    {thinking}   ← Monday's reasoning-model slide, on your bill")

# Price it like a CFO — Gemini 2.5 Flash paid tier, $ per 1M tokens
# (Monday's cheatsheet: 3.8 Flash is $0.75 / $3.75 on promo — swap the numbers and re-run)
IN_PRICE, OUT_PRICE = 0.30, 2.50
cost = tokens_in/1e6*IN_PRICE + (tokens_out + thinking)/1e6*OUT_PRICE
print(f"cost of this call at paid rates:  ${cost:.6f}")
print(f"your cost on the free tier:       $0.000000  🎉")
if cost > 0:
    print(f"\ncalls like this per $1: ~{int(1/cost):,}")


**Sit with that last number.** This is why AI features are suddenly in every product — and why sloppy, padded prompts are a real invoice line at scale. Notice the **THINKING** line, too: the model reasoned privately before it answered, and those tokens are on the invoice even though you never see them. (HW #4's tokenizer safari will make this even more concrete.)

## 🎭 Bonus — A taste of Monday: the system instruction

One parameter separates a generic chatbot from *your* product's assistant:


In [ ]:
# ── Same question, two different products ─────────────────────────
question = "Should I take out a loan to buy a food truck?"

for persona in [
    "You are a cautious bank credit officer. Answer in 2 sentences.",
    "You are an enthusiastic startup mentor. Answer in 2 sentences.",
]:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=question,
        config=types.GenerateContentConfig(system_instruction=persona, temperature=0.7),
    )
    print("🧑‍💼", persona.split('.')[0].replace('You are ', '').upper())
    print("  ", response.text.strip(), "\n")

# Same model. Same question. Different SYSTEM INSTRUCTION → different product.

## ✅ You crossed the line today

| Before 12:30 today | Now |
|---|---|
| AI user | AI **builder** |
| "the chatbot said…" | *client → model → prompt → response* |
| price = subscription | price = **tokens** (you can napkin-math any AI feature) |
| one generic assistant | any product persona, one parameter away |

### 📝 Homework #4 — The Tokenizer Safari 🧭 (due Thu Oct 1, 11:59 PM)
1. Open a tokenizer visualizer ([platform.openai.com/tokenizer](https://platform.openai.com/tokenizer))
2. Paste three things: an English sentence · the same idea in another language you know · something weird (code, emoji, names)
3. Post **one surprising screenshot** + **2 sentences on a pricing implication** in the Canvas Session 3 discussion

### 🧰 If something broke
- `userdata.get` error → the 🔑 Secrets panel: secret named exactly `GOOGLE_API_KEY`, notebook access toggled ON
- `PERMISSION_DENIED` / invalid key → re-copy the key from [aistudio.google.com/apikey](https://aistudio.google.com/apikey); no spaces
- Rate limit (`429`) → free tier allows a few calls per minute; wait 30 seconds, run again
- Anything else → re-run the install cell, then Runtime → Restart and run all

**Monday:** the CRAFT framework + you build three real mini-tools with this exact setup. *Bring your working key.* 🚀
